# fact_7 · Fonte를 웹사이트에 연결하는 임시 서버 (캐글, GPU 2개)
캐글 노트북에서 Fonte(Gemma 4 E4B + LoRA `fonte`)를 켜고, 임시 https 주소를 만들어요. 사이트 관리자 페이지에 **주소와 비밀번호**를 넣으면 LAGO 사이트의 Fonte 모드가 이 모델로 답해요.

**GPU 2개 사용 방식:** GPU마다 모델을 하나씩 올려서(일꾼 2개), 질문이 동시에 오면 놀고 있는 쪽이 맡아요. → 한 질문의 답이 빨라지는 게 아니라 **동시에 처리하는 수가 2배**예요. (4B 모델은 GPU 하나에 들어가서, 한 답을 둘이 나눠 만드는 방식은 오히려 느려져요.)

**검색 DB:** 모델 서버가 켜진 뒤 뒤에서 위키·뉴스·법령·판례 DB를 받아 올려요(fact_5와 같은 방식, 몇 분~수십 분). 올라오면 사이트가 자동으로 이 DB로 찾아요. 그 전에는 사이트가 위키백과 검색으로 대신해요.

**설정:** GPU **T4 x2** · Internet **On** · Add-ons → Secrets에 `HF_TOKEN` 체크 (`MODEL_KEY`를 Secrets에 넣어 두면 매번 같은 비밀번호를 써요)

**주의**
- 이 노트북이 켜져 있는 동안만 작동해요 (캐글 세션 최대 9~12시간). 다시 켜면 주소가 바뀌어요 → 관리자에 다시 입력.
- 모델은 `fact_5_LAGO_물어보기.ipynb`와 같은 방식(LoRA 폴더를 그대로 열기)으로 불러와요.
- 테스트가 끝나면 꼭 Stop Session (GPU 2개라 시간이 빨리 닳을 수 있어요, 캐글 프로필에서 확인).
- 이 노트북은 캐글에서 아직 실행해 보지 못한 상태예요. 오류가 나면 로그를 알려 주세요.

In [ ]:
!pip -q install fastapi uvicorn nest_asyncio httpx
!pip -q install unsloth faiss-cpu
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared

In [ ]:
import os, secrets, torch
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
HF_TOKEN = sec.get_secret("HF_TOKEN"); os.environ["HF_TOKEN"] = HF_TOKEN
try:
    KEY = sec.get_secret("MODEL_KEY")          # Secrets에 넣어 두면 매번 같은 비밀번호
except Exception:
    KEY = secrets.token_urlsafe(24)            # 없으면 이번에만 쓰는 새 비밀번호

from huggingface_hub import login, whoami, snapshot_download
login(HF_TOKEN); HF_USER = whoami()["name"]
MODELS_REPO = f"{HF_USER}/LAGO_AI_MODELS"
LORA_NAME   = "fonte"                    # fact_5와 같게 (fact_2의 LORA_NAME)
# LoRA를 한 번만 받아 두고, 일꾼들이 같은 폴더를 써요 (fact_5와 같은 방식)
p = snapshot_download(MODELS_REPO, allow_patterns=[f"{LORA_NAME}/*"], ignore_patterns=[f"{LORA_NAME}/last-checkpoint/*"], local_dir="/kaggle/working/lora")
LORA_PATH = os.path.join(p, LORA_NAME)
N_GPU = max(1, torch.cuda.device_count())
print("GPU 개수:", N_GPU, "| LoRA:", LORA_PATH)

In [ ]:
%%writefile worker.py
import os, sys, secrets, threading, torch
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
import uvicorn
PORT = int(sys.argv[1]); KEY = os.environ["MODEL_KEY"]; LORA_PATH = os.environ["LORA_PATH"]
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template
# fact_5와 똑같이 불러오기: LoRA 폴더를 그대로 모델로 열어요
model, tokenizer = FastModel.from_pretrained(LORA_PATH, max_seq_length=4096, load_in_4bit=True)
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")
FastModel.for_inference(model)
tok = getattr(tokenizer, "tokenizer", tokenizer)
app = FastAPI(); lock = threading.Lock()
class Req(BaseModel):
    system: str
    user: str
    max_new_tokens: int = 400
def generate_text(msgs, max_new_tokens):
    try:
        ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True, return_tensors="pt", return_dict=True)
    except Exception:
        m2 = [{"role": m["role"], "content": [{"type": "text", "text": m["content"]}]} for m in msgs]
        ids = tokenizer.apply_chat_template(m2, add_generation_prompt=True, tokenize=True, return_tensors="pt", return_dict=True)
    ids = ids.to(model.device)
    with lock, torch.no_grad():
        o = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(o[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()
@app.get("/health")
def health(): return {"ok": True}
@app.post("/generate")
def generate(r: Req, x_key: str = Header(default="")):
    if not secrets.compare_digest(x_key, KEY): raise HTTPException(401, "bad key")
    msgs = [{"role": "system", "content": r.system}, {"role": "user", "content": r.user}]
    return {"answer": generate_text(msgs, min(r.max_new_tokens, 600))}
print("worker ready", PORT, flush=True)
uvicorn.run(app, host="127.0.0.1", port=PORT, log_level="warning")

In [ ]:
import subprocess, time, requests, os
workers = []   # (포트, 프로세스)
for i in range(N_GPU):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i), MODEL_KEY=KEY, HF_TOKEN=HF_TOKEN, LORA_PATH=LORA_PATH)
    port = 8101 + i
    p = subprocess.Popen(["python", "worker.py", str(port)], env=env, stdout=open(f"worker{i}.log", "w"), stderr=subprocess.STDOUT)
    workers.append((port, p))
# 모델 올리는 데 몇 분 걸려요 (GPU마다 동시에 올라가요)
for port, p in workers:
    for _ in range(120):
        try:
            if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).ok: break
        except Exception: pass
        if p.poll() is not None: raise RuntimeError(f"일꾼 {port}이 꺼졌어요. worker{port-8101}.log 를 확인하세요")
        time.sleep(5)
    else:
        raise RuntimeError(f"일꾼 {port}이 시간 안에 안 켜졌어요")
print("모델 준비 완료 · 일꾼", len(workers), "개")

In [ ]:
# 문 앞 안내 서버: 놀고 있는 일꾼에게 질문을 넘겨 줘요
import threading, httpx, uvicorn, nest_asyncio, asyncio, re
from fastapi import FastAPI, Header, HTTPException, Request
nest_asyncio.apply()
gate = FastAPI()
busy = {port: 0 for port, _ in workers}
mu = threading.Lock()
DB_READY = False          # 검색 DB가 다 올라오면 True (아래 셀에서)

async def call_worker(body):
    with mu:
        port = min(busy, key=busy.get); busy[port] += 1
    try:
        async with httpx.AsyncClient(timeout=120) as c:
            r = await c.post(f"http://127.0.0.1:{port}/generate", json=body, headers={"x-key": KEY})
        if r.status_code != 200:
            raise HTTPException(502, "worker error")
        return r.json()
    finally:
        with mu: busy[port] -= 1

@gate.get("/health")
def health():
    return {"ok": True, "workers": len(workers), "db": DB_READY}

@gate.post("/generate")
async def generate(req: Request, x_key: str = Header(default="")):
    if not secrets.compare_digest(x_key, KEY):
        raise HTTPException(401, "bad key")
    return await call_worker(await req.json())

@gate.post("/ask")
async def ask_route(req: Request, x_key: str = Header(default="")):
    # 질문 하나를 받아 → 검색 DB(위키·뉴스·법령 …)에서 참고자료를 찾고 → Fonte가 답해요 (fact_5와 같은 방식)
    if not secrets.compare_digest(x_key, KEY):
        raise HTTPException(401, "bad key")
    if not DB_READY:
        raise HTTPException(503, "db loading")
    b = await req.json()
    q = str(b.get("q", "")).strip()[:500]
    if not q:
        raise HTTPException(400, "empty")
    refs = await asyncio.to_thread(find_refs, q)
    ref_txt = "\n".join(f"[{i + 1}] {r['text']}" for i, r in enumerate(refs)) or "(검색된 자료 없음)"
    out = await call_worker({"system": SYS_FACT, "user": f"참고자료:\n{ref_txt}\n\n질문: {q}", "max_new_tokens": int(b.get("max_new_tokens", 400))})
    return {"answer": out["answer"], "refs": [{"n": i + 1, "title": r.get("title", ""), "url": r.get("url", ""), "db": r["db"],
                                                "date": (r.get("date") or "")[:10], "publisher": r.get("publisher") or ""} for i, r in enumerate(refs)]}

threading.Thread(target=lambda: uvicorn.run(gate, host="127.0.0.1", port=8000, log_level="warning"), daemon=True).start()
time.sleep(3)

In [ ]:
# 검색 DB (위키 · 뉴스 · 법령 · 판례 …) 올리기 — fact_5와 같은 방식. 모델 서버는 이미 켜져 있고, DB는 뒤에서 올라와요(몇 분~수십 분).
import os, json, sqlite3, glob, numpy as np
from huggingface_hub import hf_hub_download

DB_REPO   = f"{HF_USER}/LAGO_AI_DB"
EXTRA_DBS = ["infobox", "law", "precedent", "news", "dict", "stats", "stats_catalog", "policy"]   # 있는 것만 받음
TOP_K, MIN_SCORE = 5, 0.25
DB_DIR = "/kaggle/working/lago_db"
DB_LABEL = {"infobox": "위키 정보 상자", "law": "법령", "precedent": "판례", "news": "뉴스", "dict": "우리말샘", "stats": "통계", "stats_catalog": "KOSIS 통계표", "policy": "정책뉴스"}
RECENT_RE = re.compile(r"(오늘|어제|최근|요즘|이번\s*주|이번\s*달|올해|요새|최신|속보|뉴스)")
SYS_FACT = ("[MODE:FACT] 너는 LAGO AI다. 참고자료에 근거해서 답하고, 근거 번호를 [1]처럼 붙여라. "
            "자료에 없어도 누구나 아는 일반 상식이면 짧게 답하고 끝에 '(일반 상식, 출처 없음)'을 붙여라. "
            "특정 인물·숫자·날짜·사건·최신 정보처럼 확인이 필요한 내용이 자료에 없으면 "
            "'제공된 자료에는 해당 내용이 없습니다'라고 답하라. URL을 직접 쓰지 마라.")   # fact_2·fact_5와 똑같이
DBS = {}; emb = None; emb_lock = threading.Lock()

def load_db():
    global DB_READY, emb
    try:
        t0 = time.time()
        ver = json.load(open(hf_hub_download(DB_REPO, "current.json", repo_type="dataset")))["version"]
        pats = [f"{ver}/*.faiss", f"{ver}/*.sqlite", f"{ver}/config.json"]
        pats += [f"{n}/{n}.faiss" for n in EXTRA_DBS] + [f"{n}/{n}.sqlite" for n in EXTRA_DBS]
        snapshot_download(DB_REPO, repo_type="dataset", local_dir=DB_DIR, allow_patterns=pats)
        print(f"[DB] 받기 완료 (위키 {ver}, {(time.time() - t0) / 60:.0f}분)", flush=True)
        import faiss
        from sentence_transformers import SentenceTransformer
        files = sorted(glob.glob(f"{DB_DIR}/{ver}/wiki_*.faiss")) + [f for n in EXTRA_DBS for f in glob.glob(f"{DB_DIR}/{n}/{n}.faiss")]
        for fp in files:
            name = os.path.basename(fp)[:-6]
            con = sqlite3.connect(fp[:-6] + ".sqlite", check_same_thread=False)
            cols = [r[1] for r in con.execute("PRAGMA table_info(chunks)")]
            DBS[name] = (faiss.read_index(fp), con, [c for c in ("source", "title", "url", "text", "date", "publisher") if c in cols])
            print(f"[DB]   {name}: {DBS[name][0].ntotal:,}조각", flush=True)
        emb = SentenceTransformer("google/embeddinggemma-300m", device="cuda:0", truncate_dim=256)
        DB_READY = True
        print("[DB] 검색 준비 완료 — 이제 /ask 로 위키·뉴스·법령까지 찾아요", flush=True)
    except Exception as e:
        print("[DB] 올리지 못했어요 (모델 서버는 그대로 작동해요, 사이트는 위키백과 검색으로 대신해요):", repr(e), flush=True)

def search(q, k=TOP_K, only=None):
    with emb_lock:
        qv = emb.encode_query([q], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    hits = []
    for name, (idx, con, cols) in DBS.items():
        if only and not any(name.startswith(o) for o in only):
            continue
        sc, ids = idx.search(qv, k)
        for s, i in zip(sc[0], ids[0]):
            if i >= 0 and s >= MIN_SCORE:
                row = con.execute(f"SELECT {', '.join(cols)} FROM chunks WHERE id=?", (int(i),)).fetchone()
                if row:
                    h = dict(zip(cols, row)); h["score"] = float(s)
                    h["db"] = "위키" if name.startswith("wiki") else DB_LABEL.get(name, name)
                    hits.append(h)
    hits.sort(key=lambda h: -h["score"])
    out, per = [], {}
    for h in hits:                                   # 같은 문서에서 최대 2조각
        t = h.get("title", "")
        if per.get(t, 0) >= 2: continue
        per[t] = per.get(t, 0) + 1; out.append(h)
        if len(out) >= k: break
    return out

def find_refs(q):
    refs = search(q)
    if RECENT_RE.search(q):                          # '오늘·최근' 질문은 뉴스를 최신순으로 앞에
        news = sorted(search(q, k=8, only=["news"]), key=lambda h: h.get("date") or "", reverse=True)[:3]
        refs = news + [r for r in refs if r["db"] != "뉴스"][:TOP_K - len(news)]
    return refs

threading.Thread(target=load_db, daemon=True).start()
print("검색 DB를 뒤에서 올리는 중… (아래 [DB] 로그를 보세요)")

In [ ]:
import re
tun = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"], stderr=subprocess.PIPE, text=True)
URL = None
for line in tun.stderr:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        URL = m.group(0); break
print("=" * 60)
print("관리자 페이지 → 키와 설정에 아래 두 값을 넣으세요")
print("모델 서버 주소 (MODEL_URL):", URL)
print("모델 서버 비밀번호 (MODEL_KEY):", KEY)
print("=" * 60)
# 새 주소가 인터넷에 퍼지는 데 몇 초~1분 걸려요 → 될 때까지 다시 확인 (안 돼도 멈추지 않아요)
for i in range(18):
    try:
        print("건강 확인:", requests.get(URL + "/health", timeout=10).text); break
    except Exception:
        time.sleep(5)
else:
    print("(아직 주소가 안 퍼졌어요. 1~2분 뒤 브라우저에서 주소/health 를 열어 {\"ok\":true,...} 가 보이면 정상이에요)")

In [ ]:
# 이 셀을 계속 켜 두세요 (멈추면 사이트에서 답이 안 와요)
import time
while True:
    time.sleep(300)
    print("살아 있어요", time.strftime("%H:%M:%S"), "· 처리 중:", dict(busy), "· 검색 DB:", "준비됨" if DB_READY else "올리는 중/없음")